In [ ]:
!pip install onnxruntime
!pip install onnx


KeyboardInterrupt: 

In [ ]:
!pip install optimum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 5.5 MB/s eta 0:00:00


In [ ]:
!pip install optimum[onnxruntime]
!pip install -q rank-bm25 scikit-learn


In [ ]:
import json
import pandas as pd
from sentence_transformers import InputExample, SentenceTransformer, losses, models
from torch.utils.data import DataLoader
from datetime import datetime

# 1. Ваши данные (из предыдущего контекста)
qa_json = [
  {"question": "Какой штраф уплачивает Продавец, если Товар не соответствует требованиям к упаковке и маркировке при доставке на склад Компании?", "answer": "Штраф в размере, предусмотренном Перечнем штрафов."},
  {"question": "Как Компания осуществляет приемку товара по количеству мест на складе?", "answer": "Путем сканирования штрих-кода поставки, размещенного на тарных местах (паллетах)."},
  {"question": "В какой момент формируется транспортная накладная при приемке товара Компанией?", "answer": "По итогам приемки по количеству мест (паллет)."},
  {"question": "Какой штраф выплачивает Компания Продавцу при необоснованном отказе в приемке товара по количеству мест?", "answer": "25 000 рублей."},
  {"question": "В течение какого срока Компания проводит приемку товара по количеству единиц и ассортименту?", "answer": "В течение 10 рабочих дней с даты приемки по количеству мест."},
  {"question": "Когда начинается оплата хранения товара Компанией, если сроки приемки превышены?", "answer": "С момента, когда соответствующая единица Товара была принята и указана в Акте приемки."},
  {"question": "Что происходит с товаром, не соответствующим требованиям маркировки, выявленным в ходе приемки?", "answer": "Он возвращается Продавцу на указанный в Личном кабинете пункт выдачи заказов."},
  {"question": "Сколько дней у Продавца есть для направления мотивированных возражений на Акт приемки Товаров?", "answer": "7 дней со дня публикации Акта на Портале."},
  {"question": "Что происходит, если Продавец не направил возражения на Акт приемки в установленный срок?", "answer": "Акт признается согласованным со стороны Продавца."},
  {"question": "Что вправе сделать Компания при выборочном вскрытии тары во время приемки?", "answer": "Проверить наличие грызунов, насекомых, птиц и других вредителей, а также следов их пребывания."},
  {"question": "Какое последствие наступает при неоднократном выявлении вредителей в таре (3 и более случаев)?", "answer": "Личный кабинет Продавца на Портале блокируется."},
  {"question": "Через какие ворота должна осуществляться передача крупной бытовой техники и сверхгабаритного товара?", "answer": "Только через ворота, указанные на Портале для такого Товара."},
  {"question": "Что является достаточным подтверждением полномочий лица на приемку товара на складе Продавца (DBW)?", "answer": "Наличие авторизованного доступа в мобильное приложение Компании."},
  {"question": "Когда Продавец вправе передать товар лицу, принимающему его на складе DBW?", "answer": "Только после того, как лицо считает маркировку на Товаре с помощью приложения."},
  {"question": "Как подтверждается приемка товара при передаче в ПВЗ (DBS)?", "answer": "Авторизованным доступом принимающего лица в автоматизированную систему ПВЗ и сканированием маркировки."},
  {"question": "В какой срок Компания направляет Продавцу электронный акт возврата после передачи товара в ПВЗ?", "answer": "В течение 30 календарных дней."},
  {"question": "Сколько рабочих дней дается Продавцу на направление возражений по акту возврата?", "answer": "3 рабочих дня с момента размещения акта на Портале."},
  {"question": "Кто обязан самостоятельно отслеживать информацию о возврате товара?", "answer": "Продавец."},
  {"question": "Какой максимальный объем товара можно вернуть через один пункт выдачи заказов по заявке?", "answer": "3 кубометра."},
  {"question": "Кто оплачивает доставку товара, объем которого превышает 3 кубометра?", "answer": "Продавец оплачивает доставку до ПВЗ и обратно на склад."},
  {"question": "Возвращаются ли крупногабаритные товары через пункт выдачи заказов?", "answer": "Нет, возврат крупногабаритных товаров через ПВЗ не производится."},
  {"question": "Когда услуги по доставке считаются принятыми Продавцом автоматически?", "answer": "При не подписании УПД и отсутствии замечаний в течение 3 рабочих дней."},
  {"question": "Как Продавец может подтвердить право на получение возвращенного товара?", "answer": "Документами или кодом (QR/числовым), направленным через Портал."},
  {"question": "В течение какого срока Компания передает возвращенный покупателем товар Продавцу?", "answer": "В течение 72 часов с момента передачи или отказа покупателя."},
  {"question": "Является ли добровольная компенсация расходов по возврату признанием вины Компании?", "answer": "Нет, это мера поддержки лояльности."},
  {"question": "Через сколько дней Товар признается утраченным, если Компания не вернула его после требования?", "answer": "По истечении 60 дней с момента предъявления требования."},
  {"question": "Какой срок установлен для признания товара утраченным после сообщения об инвентаризации?", "answer": "90 дней с момента сообщения."},
  {"message": "Как Компания может сообщить об утрате товара до истечения установленных сроков?", "answer": "Путем размещения сведений в Личном кабинете, Отчете о реализации или ежедневных отчетах."},
  {"question": "Возмещает ли Компания упущенную выгоду при утрате товара?", "answer": "Нет, упущенная выгода и косвенные убытки не подлежат возмещению."},
  {"question": "По какой формуле рассчитывается компенсация за утраченный товар (Св)?", "answer": "Св = Ц - НДС - К - (Ц - НДС - К) * Нац%"},
  {"question": "Как определяется Потенциальная цена (Ц) для расчета компенсации, если товар продан менее 10 раз за год?", "answer": "Как среднее значение цен на товары того же Предмета за аналогичный период."},
  {"question": "Что произойдет, если рассчитанная Потенциальная цена превысит Максимальную цену по Предмету?", "answer": "Потенциальная цена приравнивается к Максимальной цене по Предмету."},
  {"question": "Какие документы обязан предоставить Продавец для подтверждения себестоимости утраченного товара?", "answer": "Договоры, накладные, акты, УПД, счета-фактуры и документы об оплате."},
  {"question": "В течение какого срока Продавец должен предоставить документы о себестоимости по запросу?", "answer": "В течение 10 дней с момента получения запроса."},
  {"question": "В течение какого срока Компания выплачивает компенсацию за утрату?", "answer": "В течение 30 рабочих дней после истечения срока возврата или предоставления документов."},
  {"question": "Когда обязанность Компании по возмещению ущерба считается исполненной?", "answer": "С момента отражения суммы в Личном кабинете как доступной к востребованию."},
  {"question": "В течение какого срока Компания перечисляет истребованную компенсацию на счет?", "answer": "В течение 5 рабочих дней с момента востребования."},
  {"question": "К какому событию не применяются общие правила компенсации за утрату (пп. 11.3.1–11.3.9)?", "answer": "К утрате товара в результате пожара 13.01.2024 на складе в Шушарах."},
  {"question": "Что обязан сделать Продавец, если получил страховую выплату за утраченный товар, за который Компания уже возместила ущерб?", "answer": "Возвратить Компании выплаченное возмещение в течение 10 дней."},
  {"question": "Что вправе сделать Компания, если обнаружит товар, за который уже была выплачена компенсация?", "answer": "Отразить товар в Личном кабинете с удержанием ранее выплаченной суммы."},
  {"question": "Как Компания сообщает об утрате товара при пожаре 13.01.2024?", "answer": "Размещением Отчета о компенсации в формате .xls на специальной странице Портала."},
  {"question": "Что означает принятие Продавцом Отчета о компенсации по пожару?", "answer": "Заключение соглашения о размере ущерба и отказ от иных претензий к Компании."},
  {"question": "Как Продавец может востребовать часть компенсации, если она отражена частями?", "answer": "Нажав кнопку «Вывести» в Личном кабинете после появления доступной суммы."},
  {"question": "Почему сумма к востребованию может быть меньше согласованной компенсации?", "answer": "Из-за сальдирования с текущими требованиями Компании к Продавцу."},
  {"question": "Как рассчитать Коэффициент проблемного товара (Кпт)?", "answer": "Кпт = (ПТ / Т) * 100%"},
  {"question": "Что обозначает переменная ПТ в формуле проблемного товара?", "answer": "Количество единиц с дефектами, некомплектностью, пересортицей или иными недостатками."},
  {"question": "Что обозначает переменная Т в формуле проблемного товара?", "answer": "Количество принятых покупателями единиц, за исключением возвращенных."},
  {"question": "За какой период рассчитывается Коэффициент проблемного товара?", "answer": "За 90 дней, предшествующих дню расчета."},
  {"question": "Где публикуется Коэффициент проблемного товара?", "answer": "В Личном кабинете Продавца на Портале."},
  {"question": "Какие дефекты учитываются при расчете проблемного товара?", "answer": "Повреждения тары/упаковки, некомплектность, пересортица, несоответствие карточке товара и иные дефекты."},
  {"question": "Обязан ли Продавец ознакомить своих сотрудников с правилами пропускного режима на территории Компании?", "answer": "Да, и обеспечить их соблюдение."},
  {"question": "Что должен сделать Продавец при фиксации противоправных действий его работниками на территории?", "answer": "Представить информацию о нарушителях и оказать содействие в привлечении их к ответственности."},
  {"question": "Какими способами может быть зафиксировано правонарушение на территории Компании?", "answer": "Техническими средствами (камеры) или свидетельскими показаниями/проверками."},
  {"question": "Может ли Компания в одностороннем порядке уменьшить задолженность на стоимость доставки/сборки?", "answer": "Да, если Продавец не подписал УПД и не направил замечания за 3 дня."},
  {"question": "Кто несет ответственность за изменение статуса передачи товара на Портале при DBW?", "answer": "Продавец обязан убедиться в изменении статуса и сообщить об ошибке."},
  {"question": "Куда возвращается товар при выявлении нарушений маркировки в ходе приемки?", "answer": "На пункт выдачи заказов, указанный Продавцом для возвратов."},
  {"question": "Что происходит с актом приемки, если Продавец предъявляет возражения?", "answer": "Стороны согласовывают Акт в окончательной редакции."},
  {"question": "Кто составляет акт осмотра при вскрытии тары на наличие вредителей?", "answer": "Компания, сопровождая действия фото- и видеосъемкой."},
  {"question": "Какой документ формируется после приемки по количеству паллет?", "answer": "Транспортная накладная."},
  {"question": "Где Продавец выбирает адреса для возврата товаров?", "answer": "Сразу после заключения Договора, выбирая из списка на Портале."},
  {"question": "Какой адрес ПВЗ Компания учитывает в первую очередь при возврате?", "answer": "Основной адрес ПВЗ, указанный Продавцом."},
  {"question": "Где указывается информация о дате поступления возвращаемого товара?", "answer": "В Отчете по возвратам и перемещению товаров в Личном кабинете."},
  {"question": "Что происходит с товаром, объем возврата которого превышает 3 кубометра?", "answer": "Он возвращается на склад, а доставку оплачивает Продавец."},
  {"question": "Когда производится приемка возвращенного товара по количеству и качеству?", "answer": "В момент передачи на Пункте Выдачи Заказов."},
  {"question": "Как Продавец может установить максимальное число доставок до возврата товара?", "answer": "Через функционал Портала, что приравнивается к заявке на возврат."},
  {"question": "Считается ли размещение отчета о компенсации за пожар признанием утраты товара?", "answer": "Нет, это только предложение о возмещении."},
  {"question": "Какие документы нужны для пересчета компенсации при несогласии с отчетом по пожару?", "answer": "Подтверждающие стоимость приобретения или производства товара."},
  {"question": "Может ли Компания запросить дополнительные документы для подтверждения расходов при пожаре?", "answer": "Да, вправе запросить дополнительные документы."},
  {"question": "В каком случае Компенсация по утрате выплачивается на основании себестоимости?", "answer": "Если Компания соглашается с предоставленными документами о расходах."},
  {"question": "Покрывает ли компенсация по себестоимости все убытки Продавца?", "answer": "Да, покрывает все убытки, связанные с утратой."},
  {"question": "Каким способом Продавец обязан востребовать компенсацию?", "answer": "Только с использованием Личного кабинета на Портале."},
  {"question": "Что происходит, если Продавец не согласен с Актом приемки и не направляет возражения?", "answer": "Акт считается согласованным автоматически."},
  {"question": "Какой статус должен отобразиться на Портале после передачи товара на складе DBW?", "answer": "Статус, указывающий, что передача Товара совершена."},
  {"question": "Что обязан сделать Продавец, если статус передачи товара не изменился?", "answer": "Незамедлительно передать информацию об этом в Компанию."},
  {"question": "В каком случае возврат товара в ПВЗ не производится?", "answer": "Для крупногабаритных товаров."},
  {"question": "Как определяется стоимость доставки возвращаемого товара?", "answer": "На основании Тарифов, размещенных на Портале."},
  {"question": "Кто подтверждает полномочия лица на приемку товара в ПВЗ?", "answer": "Авторизованный доступ в систему ПВЗ и сканирование маркировки."},
  {"question": "Где Продавец принимает возвращенный покупателем товар при DBS?", "answer": "В том ПВЗ, где покупатель получал товар."},
  {"question": "Где Продавец принимает товар при отказе покупателя от приемки (DBS)?", "answer": "В том ПВЗ, где Продавец передал товар Компании."},
  {"question": "Какой формулой определяется размер НДС в расчете компенсации?", "answer": "НДС = Ц * НДС% / (100% + НДС%)"},
  {"question": "От чего рассчитывается надбавка (Нац%) в формуле компенсации?", "answer": "От потенциальной цены за вычетом Комиссии и НДС."},
  {"question": "За какой период усредняется цена при расчете Потенциальной цены?", "answer": "За 365 дней, предшествующих дате последней продажи."},
  {"question": "Где размещен перечень предметов для расчета средней цены同类 товаров?", "answer": "На Портале seller.wildberries.ru/dynamic-product-categories/commission (в системе Компании)."},
  {"question": "Куда направляется мотивированное несогласие с Актом приемки?", "answer": "На Портал."},
  {"question": "Что включает в себя доступная к востребованию сумма на Портале?", "answer": "Компенсацию и выручку за минусом требований Компании к Продавцу."},
  {"question": "Нарушает ли Компания обязательства, если сумма к выплате меньше компенсации из-за взаиморасчетов?", "answer": "Нет, не признается нарушившим обязанность."},
  {"question": "Где можно найти инструкцию по применению правил пропускного режима?", "answer": "В инструкции seller.wildberries.ru/instructions/ru/ru/material/A-998."},
  {"question": "Кто несет ответственность за нарушения режима третьими лицами, привлеченными Продавцом?", "answer": "Продавец."},
  {"question": "Что является основанием для блокировки Личного кабинета?", "answer": "3 и более случая обнаружения вредителей в таре."},
  {"question": "Кто вправе в любое время заявить о возврате товара?", "answer": "Компания."},
  {"question": "Кто вправе в любое время потребовать возврата нереализованного товара?", "answer": "Продавец, направив заявку через Портал."},
  {"question": "Какой срок отводится на согласование Акта приемки при наличии возражений?", "answer": "Стороны согласовывают его в окончательной редакции (срок не фиксирован в пункте)."},
  {"question": "Когда Компания составляет Акт приемки при DBS в ПВЗ?", "answer": "По результатам приемки Товара в ПВЗ."},
  {"question": "Кто несет риск утраты товара до момента изменения статуса на Портале?", "answer": "Продавец обязан контролировать статус и сообщить о сбое."},
  {"question": "Что происходит с актом возврата при отсутствии возражений?", "answer": "Признается утвержденным Продавцом."},
  {"question": "Какой документ является основанием для выплаты компенсации по пожару?", "answer": "Отчет о компенсации в формате .xls, размещенный на специальной странице."},
  {"question": "Может ли Компания компенсировать ущерб до истечения 60 дней?", "answer": "Да, если сообщит об утрате или необходимости инвентаризации."},
  {"question": "Кто определяет адрес передачи возвращаемого товара?", "answer": "Продавец выбирает из списка на Портале."},
  {"question": "Что происходит, если Компания находит товар, за который уже выплачена компенсация?", "answer": "Товар отражается в ЛК, а сумма компенсации удерживается с Баланса."},
  {"question": "Какие действия Компании фиксируются при проверке на вредителей?", "answer": "Составляется акт осмотра, производится фото- и видеосъемка."}
]

# 2. Подготовка датасета для Sentence Transformers
# Используем подход: Вопрос и Ответ должны быть близки в векторном пространстве
train_examples = []
for item in qa_json:
    q = item.get('question', item.get('message', '')) # Обработка ключа 'message' в одном из примеров
    a = item['answer']
    train_examples.append(InputExample(texts=[q, a]))

print(f"✅ Подготовлено {len(train_examples)} пар Вопрос-Ответ")

# 3. Разделение на Train/Test (Глава 7: Схемы валидации)
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(train_examples, test_size=0.1, random_state=42)
print(f"📊 Train: {len(train_data)}, Test: {len(test_data)}")

✅ Подготовлено 100 пар Вопрос-Ответ
📊 Train: 90, Test: 10


In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch


# 🛡️ Защита от случайного запуска с пустыми данными
if not isinstance(qa_json, list) or len(qa_json) == 0:
    raise ValueError("❌ qa_json пуст или не является списком. Вставьте ваши данные перед запуском.")

# 1. Загрузка модели
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SentenceTransformer('./models/rubert_finetuned', device=device)

# 2. Безопасное извлечение ответов (учитывает ключ 'message', если он встретится)
answers = [item.get('answer', '') for item in qa_json]

# 3. Предвычисление эмбеддингов (делается 1 раз при старте)
print("🧠 Кодирование базы ответов...")
answer_embeddings = model.encode(answers, convert_to_tensor=True, batch_size=32, show_progress_bar=True)

def find_answer(query: str, top_k: int = 1, min_score: float = 0.60) -> list:
    """Поиск наиболее релевантного ответа с проверкой уверенности"""
    query_embedding = model.encode(query, convert_to_tensor=True)

    # Косинусное сходство
    similarities = util.cos_sim(query_embedding, answer_embeddings)[0]

    # Топ-k результатов
    top_k = min(top_k, len(similarities))
    top_scores, top_indices = torch.topk(similarities, k=top_k)

    results = []
    for score, idx in zip(top_scores, top_indices):
        results.append({
            'answer': qa_json[idx]['answer'],
            'score': float(score),
            'question': qa_json[idx].get('question', qa_json[idx].get('message', 'Нет вопроса'))
        })

    # 🔒 Если уверенность ниже порога → возвращаем заглушку
    if results and results[0]['score'] < min_score:
        return [{'answer': '🤷‍♂️ Не нашёл релевантного ответа. Переформулируйте вопрос.', 'score': results[0]['score']}]

    return results

# 🧪 Тест
result = find_answer("как сделать возврат?")
print(f"💡 Ответ: {result[0]['answer']}")
print(f"📊 Уверенность: {result[0]['score']:.3f}")

🧠 Кодирование базы ответов...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

💡 Ответ: 🤷‍♂️ Не нашёл релевантного ответа. Переформулируйте вопрос.
📊 Уверенность: 0.554


In [ ]:
result = find_answer("павпввс")
print(f"💡 Ответ: {result[0]['answer']}")
print(f"📊 Уверенность: {result[0]['score']:.3f}")

💡 Ответ: 🤷‍♂️ Не нашёл релевантного ответа. Переформулируйте вопрос.
📊 Уверенность: 0.463


In [ ]:
# 🔧 Увеличьте batch_size для лучшего негативного семплинга
# Если память позволяет: batch_size=32 или 64
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=32)

# 🔧 Добавьте оценку на тесте
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# Подготовьте corpus и queries для оценки
# 🔧 Безопасное извлечение вопроса (поддерживает 'question' и 'message')
def get_question(item):
    return item.get('question') or item.get('message') or ''

# Подготовка данных для evaluator
corpus = {i: item['answer'] for i, item in enumerate(qa_json)}
queries = {i: get_question(item) for i, item in enumerate(qa_json)}

# Для тестовых данных — аналогично
test_queries = {i: get_question(item) for i, item in enumerate(test_data)}
test_corpus = {i: item['answer'] for i, item in enumerate(test_data)}

# Создайте evaluator (только для тестовых данных!)
evaluator = InformationRetrievalEvaluator(
    queries={i: qa_json[i]['question'] for i in range(len(test_data))},
    corpus={i: qa_json[i]['answer'] for i in range(len(qa_json))},
    relevant_docs={i: {i} for i in range(len(test_data))}  # Упрощённо: вопрос → свой ответ
)

# Добавьте evaluator в fit()
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    evaluator=evaluator,
    evaluation_steps=100,  # Оценка каждые 100 шагов
    epochs=num_epochs,
    warmup_steps=warmup_steps,
    output_path='./models/rubert_finetuned',
    show_progress_bar=True
)

AttributeError: 'InputExample' object has no attribute 'get'

In [ ]:
from optimum.exporters.onnx import main_export
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

model_path = "./models/ru_qa_finetuned"
output_path = "./models/ru_qa_finetuned_onnx"

# 1. Явно загружаем компоненты локально (Chapter 10: Reproducibility)
# local_files_only=True запрещает обращение к Hub
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)

# 2. Экспорт
# Передаем путь, но убедимся, что он существует
if not os.path.exists(output_path):
    os.makedirs(output_path)

try:
    main_export(
        model_name_or_path=model_path,
        output=output_path,
        task="sequence-classification",
        opset=14,
        # trust_remote_code=True может потребоваться для кастомных архитектур
    )
    print(f"✅ Модель успешно экспортирована в {output_path}")
except Exception as e:
    print(f"❌ Ошибка экспорта: {e}")
    # Fallback: Попробовать сохранить через torch.onnx если optimum fails (Chapter 13: Fallbacks)

Opset 14 is lower than the recommended minimum opset (18) to export transformer. The ONNX export may fail or the exported model may be suboptimal.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


❌ Ошибка экспорта: 'Unknown task: text-classification. Possible values are: `feature-extraction` for SentenceTransformer, `sentence-similarity` for SentenceTransformer'


✅ PyTorch модель загружена

--- ТЕСТИРОВАНИЕ ---
📏 PyTorch output shape: (1, 768)
📏 ONNX raw shape: (1, 6, 768)
📏 ONNX pooled shape: (768,)
🔍 Совпадение PyTorch vs ONNX: 1.00000 (должно быть > 0.99)

--- ПОИСК ОТВЕТА ---
🔍 Топ-5 кандидатов:
  1. [0.3955] В течение 10 рабочих дней с даты приемки по количеству мест....
  2. [0.3955] В течение 10 рабочих дней с даты приемки по количеству мест....
  3. [0.3955] В течение 10 рабочих дней с даты приемки по количеству мест....
  4. [0.3955] В течение 10 рабочих дней с даты приемки по количеству мест....
  5. [0.3790] Путем сканирования штрих-кода поставки, размещенного на тарных местах (паллетах)...
 Вопрос: возврат товара по браку
💡 Лучший ответ: В течение 10 рабочих дней с даты приемки по количеству мест.
📊 Score: 0.3955


In [ ]:
from google.colab import files
import shutil
import os

# Создаем zip архив со всей моделью и токенизатором
# Первый аргумент - имя архива (без расширения)
# Третий аргумент - корневая директория
# Четвертый аргумент - что именно архивировать
shutil.make_archive(
    'rubert_finetuned_onnx_model',  # имя zip файла (без .zip)
    'zip',
    './models',                       # корневая директория
    'rubert_finetuned_onnx'          # папка для архивации
)

# Скачиваем архив
files.download('rubert_finetuned_onnx_model.zip')

print("\n✅ Готово для внедрения в Go-сервис!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Готово для внедрения в Go-сервис!


In [ ]:
if not isinstance(qa_json, list) or len(qa_json) < 10:
    raise ValueError("❌ Ошибка: qa_json пуст или содержит менее 10 записей. Вставьте ваши данные!")

for i, item in enumerate(qa_json):
    if 'message' in item and 'question' not in item:
        item['question'] = item.pop('message')
    if 'question' not in item or 'answer' not in item:
        raise ValueError(f"❌ Ошибка в записи #{i}: отсутствуют ключи 'question' или 'answer'")

print(f"✅ Загружено и проверено записей: {len(qa_json)}")

# %% 3. Подготовка датасета
train_examples = [InputExample(texts=[item['question'], item['answer']]) for item in qa_json]
train_data, test_data = train_test_split(train_examples, test_size=0.15, random_state=42)

train_dataloader = DataLoader(train_data, shuffle=True, batch_size=16)
print(f"📊 Train: {len(train_data)} | Test: {len(test_data)}")

# %% 4. Обучение модели
MODEL_NAME = "DeepPavlov/rubert-base-cased"
#MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"  # ⚡ Быстрая, хорошая точность
OUTPUT_DIR = "./models/rubert_qa_finetuned"
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"📥 Загрузка базовой модели: {MODEL_NAME} (device: {DEVICE})")
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

# MultipleNegativesRankingLoss идеально подходит для пар (вопрос, ответ)
train_loss = losses.MultipleNegativesRankingLoss(model)

print("🚀 Запуск обучения...")
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=10,
    warmup_steps=int(len(train_dataloader) * 5 * 0.1),
    output_path=OUTPUT_DIR,
    show_progress_bar=True,
)
print(f"✅ Модель успешно обучена и сохранена в `{OUTPUT_DIR}`")

✅ Загружено и проверено записей: 100
📊 Train: 85 | Test: 15
📥 Загрузка базовой модели: DeepPavlov/rubert-base-cased (device: cpu)
🚀 Запуск обучения...


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss


✅ Модель успешно обучена и сохранена в `./models/rubert_qa_finetuned`


In [ ]:
model = SentenceTransformer(OUTPUT_DIR, device=DEVICE)

# Предвычисляем эмбеддинги ответов (делается 1 раз)
answers = [item['answer'] for item in qa_json]
answer_embeddings = model.encode(answers, convert_to_tensor=True, batch_size=32, show_progress_bar=True)

def find_answer(query: str, top_k: int = 1, min_score: float = 0.40) -> list:
    """Поиск релевантного ответа с порогом уверенности"""
    query_emb = model.encode(query, convert_to_tensor=True)
    similarities = util.cos_sim(query_emb, answer_embeddings)[0]

    top_k = min(top_k, len(similarities))
    top_scores, top_indices = torch.topk(similarities, k=top_k)

    results = []
    for score, idx in zip(top_scores, top_indices):
        results.append({
            'answer': qa_json[idx]['answer'],
            'score': float(score),
            'original_question': qa_json[idx]['question']
        })

    # 🔒 Если уверенность ниже порога → заглушка
    if results and results[0]['score'] < min_score:
        return [{'answer': '🤷‍♂️ Не нашёл релевантного ответа. Попробуйте переформулировать.', 'score': results[0]['score']}]

    return results


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:

from rank_bm25 import BM25Okapi
import re

# Предобработка текста
def preprocess(text):
    return re.findall(r'[\wа-яё]+', text.lower())

# Подготовка BM25 индекса
tokenized_corpus = [preprocess(item['answer']) for item in qa_json]
bm25 = BM25Okapi(tokenized_corpus)

def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.7) -> list:
    """
    Комбинирует семантический поиск (alpha) и keyword-поиск (1-alpha)
    alpha=0.7 → 70% веса на семантику, 30% на ключевые слова
    """
    # 1. Семантические скоры
    query_emb = model.encode(query, convert_to_tensor=True)
    semantic_scores = util.cos_sim(query_emb, answer_embeddings)[0].cpu().numpy()

    # 2. BM25 скоры
    query_tokens = preprocess(query)
    bm25_scores = bm25.get_scores(query_tokens)

    # 3. Нормализация + комбинация
    from sklearn.preprocessing import MinMaxScaler
    scaler = MinMaxScaler()

    sem_norm = scaler.fit_transform(semantic_scores.reshape(-1, 1)).flatten()
    bm25_norm = scaler.fit_transform(bm25_scores.reshape(-1, 1)).flatten()

    combined = alpha * sem_norm + (1 - alpha) * bm25_norm

    # 4. Топ-k результатов
    top_indices = combined.argsort()[-top_k:][::-1]

    return [{
        'answer': qa_json[idx]['answer'],
        'score': float(combined[idx]),
        'original_question': qa_json[idx]['question'],
        'semantic_score': float(semantic_scores[idx]),
        'bm25_score': float(bm25_scores[idx])
    } for idx in top_indices]

# Установка зависимости

In [ ]:
print("ДЕМОНСТРАЦИЯ РАБОТЫ:")
test_queries = [
    "Как вернуть товар без бирки?",
    "В каком случае возврат товара в ПВЗ не производится?",
    "Как Продавец может установить максимальное число доставок до возврата товара?",
    "Покрывает ли компенсация по себестоимости все убытки Продавца?",
    "Покрывает ли компенсация по себестоимости Продавца?",
    "Является ли возмещение, рассчитанное по себестоимости, достаточным для полного покрытия всех убытков, понесенных Продавцом?",
    "почему",
    "Бирки в шнутрах",
    "Кто убил марка",
    "Как оформить возврат?"
]

for q in test_queries:
    res = hybrid_search(q, top_k=3)[0]
    print(f"Вопрос: {q}")
    print(f"Ответ: {res['answer']} (score: {res['score']:.3f})")
    print("-" * 60)

ДЕМОНСТРАЦИЯ РАБОТЫ:
Вопрос: Как вернуть товар без бирки?
Ответ: Отразить товар в Личном кабинете с удержанием ранее выплаченной суммы. (score: 0.843)
------------------------------------------------------------
Вопрос: В каком случае возврат товара в ПВЗ не производится?
Ответ: Нет, возврат крупногабаритных товаров через ПВЗ не производится. (score: 1.000)
------------------------------------------------------------
Вопрос: Как Продавец может установить максимальное число доставок до возврата товара?
Ответ: Продавец выбирает из списка на Портале. (score: 0.716)
------------------------------------------------------------
Вопрос: Покрывает ли компенсация по себестоимости все убытки Продавца?
Ответ: Да, покрывает все убытки, связанные с утратой. (score: 1.000)
------------------------------------------------------------
Вопрос: Покрывает ли компенсация по себестоимости Продавца?
Ответ: Да, покрывает все убытки, связанные с утратой. (score: 1.000)
----------------------------------------

In [ ]:
# 1. Установка зависимостей

import re
import numpy as np
import torch
import onnxruntime as ort
from transformers import AutoTokenizer
from rank_bm25 import BM25Okapi
from sklearn.preprocessing import MinMaxScaler
from sentence_transformers import util

# ==========================================
# 2. ЭКСПОРТ МОДЕЛИ В ONNX
# ==========================================
# Замените на вашу модель (должна поддерживать Feature Extraction)


print("📦 Экспорт модели в ONNX...")
from optimum.onnxruntime import ORTModelForFeatureExtraction

ort_model = ORTModelForFeatureExtraction.from_pretrained(MODEL_NAME, export=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

ort_model.save_pretrained("./onnx_model/")
tokenizer.save_pretrained("./onnx_model/")
print("✅ Модель сохранена в ./onnx_model/")

# ==========================================
# 3. ONNX-ОБЁРТКА С MEAN POOLING
# ==========================================
class ONNXEmbedder:
    def __init__(self, model_dir="./onnx_model/"):
        # providers=["CPUExecutionProvider"] можно заменить на "CUDAExecutionProvider" если есть GPU
        self.session = ort.InferenceSession(f"{model_dir}/model.onnx",
                                            providers=["CPUExecutionProvider"])
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)

    def encode(self, texts, convert_to_tensor=False):
        if isinstance(texts, str):
            texts = [texts]

        inputs = self.tokenizer(texts, padding=True, truncation=True, return_tensors="np")

        # 1. Динамически собираем входы под сигнатуру ONNX
        ort_inputs = {}
        for inp in self.session.get_inputs():
            if inp.name == "input_ids":
                ort_inputs[inp.name] = inputs["input_ids"]
            elif inp.name == "attention_mask":
                ort_inputs[inp.name] = inputs["attention_mask"]
            elif inp.name == "token_type_ids":
                # Модель требует тензор, но для эмбеддингов он не важен → нули
                ort_inputs[inp.name] = np.zeros_like(inputs["input_ids"])

        # 2. Инференс
        outputs = self.session.run(None, ort_inputs)
        token_embeddings = outputs[0]  # last_hidden_state

        # 3. Attention_mask берём из токенизатора (ONNX его не возвращает)
        attention_mask = inputs["attention_mask"]

        # 4. Mean Pooling
        input_mask_expanded = np.expand_dims(attention_mask, axis=-1)
        sum_embeddings = np.sum(token_embeddings * input_mask_expanded, axis=1)
        sum_mask = np.sum(input_mask_expanded, axis=1)
        # Защита от деления на ноль для пустых строк
        sentence_embeddings = sum_embeddings / np.maximum(sum_mask, 1e-9)

        if convert_to_tensor:
            return torch.tensor(sentence_embeddings, dtype=torch.float32)
        return sentence_embeddings

# Инициализация эмбеддера
onnx_embedder = ONNXEmbedder("./onnx_model/")

# ==========================================
# 4. ПОДГОТОВКА ДАННЫХ И ГИБРИДНЫЙ ПОИСК
# ==========================================
# ⚠️ ВАЖНО: answer_embeddings должны быть посчитаны ТОЙ ЖЕ ONNX-моделью
answer_texts = [item['answer'] for item in qa_json]
answer_embeddings = onnx_embedder.encode(answer_texts, convert_to_tensor=True)

# BM25 индекс остаётся без изменений
def preprocess(text):
    return re.findall(r'[\wа-яё]+', text.lower())

tokenized_corpus = [preprocess(item['answer']) for item in qa_json]
bm25 = BM25Okapi(tokenized_corpus)

def hybrid_search(query: str, top_k: int = 5, alpha: float = 0.7) -> list:
    # 1. Семантические скоры (ONNX)
    query_emb = onnx_embedder.encode(query, convert_to_tensor=True)
    semantic_scores = util.cos_sim(query_emb, answer_embeddings)[0].cpu().numpy()

    # 2. BM25 скоры
    query_tokens = preprocess(query)
    bm25_scores = bm25.get_scores(query_tokens)

    # 3. Нормализация + комбинация
    scaler = MinMaxScaler()
    sem_norm = scaler.fit_transform(semantic_scores.reshape(-1, 1)).flatten()
    bm25_norm = scaler.fit_transform(bm25_scores.reshape(-1, 1)).flatten()

    combined = alpha * sem_norm + (1 - alpha) * bm25_norm

    # 4. Топ-k результатов
    top_indices = combined.argsort()[-top_k:][::-1]

    return [{
        'answer': qa_json[idx]['answer'],
        'score': float(combined[idx]),
        'original_question': qa_json[idx]['question'],
        'semantic_score': float(semantic_scores[idx]),
        'bm25_score': float(bm25_scores[idx])
    } for idx in top_indices]

The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 was already converted to ONNX but got `export=True`, the model will be converted to ONNX once again. Don't forget to save the resulting model with `.save_pretrained()`


📦 Экспорт модели в ONNX...
✅ Модель сохранена в ./onnx_model/


In [ ]:
ZIP_NAME = "onnx_model_bundle.zip"
MODEL_DIR = "./onnx_model"

if os.path.isdir(MODEL_DIR):
    # 1. Создаём архив (содержимое папки без самой папки)
    shutil.make_archive(ZIP_NAME.replace(".zip", ""), 'zip', root_dir=MODEL_DIR)
    print(f"✅ Архив '{ZIP_NAME}' успешно создан ({os.path.getsize(ZIP_NAME) / 1024 / 1024:.2f} MB).")

    # 2. Скачивание: авто-триггер в Colab / ссылка в Jupyter
    try:
        from google.colab import files
        files.download(ZIP_NAME)
        print("📥 Загрузка началась (Colab).")
    except ImportError:
        display(HTML(f'<a href="{ZIP_NAME}" download style="background:#4CAF50;color:white;padding:10px 15px;text-decoration:none;border-radius:5px;">📥 Скачать {ZIP_NAME}</a>'))
else:
    print(f"❌ Папка '{MODEL_DIR}' не найдена. Убедитесь, что экспорт модели прошёл успешно.")

✅ Архив 'onnx_model_bundle.zip' успешно создан (414.90 MB).


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Загрузка началась (Colab).
